# Eastern Mau Forest Loss Monitoring (2019–2024)

Satellite-based forest-cover change detection for the **Eastern Mau Forest Reserve**, Kenya, using Sentinel-2 imagery on Google Earth Engine.

This notebook establishes a forest baseline, detects canopy loss against it, and reports a conservative, validated loss figure. It is built to be **modular** — region, dates, and thresholds live in one configuration cell, so adapting it to another forest block or period means editing only that cell.

**Pipeline:** config → study area (WDPA) → cloud-masked composites → NDVI → forest classification → visual validation → baseline normalization → loss detection → noise filtering → quantification → figure export.

**Requirements:** a Google Earth Engine account and a registered Cloud project. Set your project ID in the setup cell below.

## 0. Setup and authentication

Run once per session. Replace `PROJECT_ID` with your own registered Earth Engine Cloud project.

In [1]:
# geemap provides interactive maps for Earth Engine in notebooks.
# It is pre-installed in Colab; install it elsewhere if needed.
try:
    import geemap
except ImportError:
    !pip install geemap -q
    import geemap

import ee

# --- Your registered Earth Engine Cloud project --------------
PROJECT_ID = 'mau-sentinel'   # <-- change to your project ID

ee.Authenticate(auth_mode='notebook')
ee.Initialize(project=PROJECT_ID)
print("Earth Engine is ready to go!")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=vsrtOW0j82TNLI2y8TgN6A_-w_yMXFkctmtH4X9WcxI&tc=J1nQIdv1pfhRZBx23LJGlvjux33nv6O_MnewxShdxJQ&cc=QeIjIzG_A_jhqMTaIJlvN8GB6XzyGuOTrXP4qQ9GBX0

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AdkVLPzZigkgY1qdr60mlmK9Aw7rfT5R7upbDg2EtcD1viDxJ0MKlj8NBGs

Successfully saved authorization token.
Earth Engine is ready to go!


## 1. Configuration

Everything tunable lives here. To adapt the pipeline to a different region, time period, or sensitivity, edit **only** this cell — the logic downstream reads from these variables.

In [2]:
# --- Spatial scope -------------------------------------------
# v1 studies the Eastern Mau block, selected from the WDPA
# protected-areas dataset by name (see Module 2). To study a
# different block, change this list, e.g. ['South West Mau'] or
# ['Eastern Mau', 'South West Mau'].
TARGET_PA_NAMES = ['Eastern Mau']

# --- Time window ---------------------------------------------
# Two snapshot years for a clean before/after comparison.
# We composite over the Jan-Mar dry season to minimise cloud.
YEAR_BEFORE  = 2019
YEAR_AFTER   = 2024
SEASON_START = '-01-01'   # appended to each year
SEASON_END   = '-03-31'

# --- Sensor / data -------------------------------------------
S2_COLLECTION     = 'COPERNICUS/S2_SR_HARMONIZED'
MAX_CLOUD_PERCENT = 20     # discard scenes cloudier than this

# --- Forest definition ---------------------------------------
# Forest = NDVI above this threshold on the median composite.
# Transparent and auditable; validated visually in Module 5.
NDVI_FOREST_THRESHOLD = 0.6

# --- Change detection ----------------------------------------
# Minimum NDVI drop for a pixel to count as forest loss.
# Conservative: ignores composite wobble, catches real change.
NDVI_CHANGE_THRESHOLD = 0.15

# Minimum real-world patch size kept (noise filter).
# 10 m pixels => 100 m2 each; 0.5 ha = 50 connected pixels.
MIN_MAPPING_UNIT_HA = 0.5
PIXELS_PER_HA       = 100
MIN_PATCH_PIXELS    = int(MIN_MAPPING_UNIT_HA * PIXELS_PER_HA)

# --- Visualization -------------------------------------------
NDVI_PALETTE = ['white', 'yellow', 'green']
FOREST_COLOR = '#1a9850'   # green = forest / retained
LOSS_COLOR   = '#d73027'   # red   = forest lost

true_colour_vis = {'min': 0.0, 'max': 0.3, 'bands': ['B4', 'B3', 'B2']}
ndvi_vis        = {'min': 0.0, 'max': 1.0, 'palette': NDVI_PALETTE}

print("Config loaded.")
print(f"  Region : {', '.join(TARGET_PA_NAMES)}")
print(f"  Years  : {YEAR_BEFORE} vs {YEAR_AFTER}")
print(f"  Forest : NDVI > {NDVI_FOREST_THRESHOLD}")

Config loaded.
  Region : Eastern Mau
  Years  : 2019 vs 2024
  Forest : NDVI > 0.6


## 2. Study area from WDPA

The boundary comes from the **World Database on Protected Areas** (WDPA, UNEP-WCMC) — an authoritative source — selected by name rather than an arbitrary bounding box.

*WDPA is free for non-commercial, research, and educational use. Commercial use requires written permission from UNEP-WCMC.*

In [3]:
wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons")
kenya_pas = wdpa.filter(ee.Filter.eq('ISO3', 'KEN'))

def build_aoi_from_wdpa(names):
    """Return (geometry, features) for the named WDPA polygons."""
    selected = kenya_pas.filter(ee.Filter.inList('NAME', names))
    return selected.geometry(), selected

aoi, aoi_features = build_aoi_from_wdpa(TARGET_PA_NAMES)
print(f"AOI set to: {', '.join(TARGET_PA_NAMES)}")

AOI set to: Eastern Mau


## 3. Cloud-masked Sentinel-2 composites

For each year we pull every dry-season scene over the AOI under the cloud limit, mask residual clouds with the QA60 band, and take the **per-pixel median**. The median rejects transient clouds without us having to pick a single "good" image.

In [4]:
def mask_s2_clouds(image):
    """Mask cloud and cirrus pixels using the QA60 band; scale to 0-1."""
    qa = image.select('QA60')
    cloud_bit  = 1 << 10   # opaque clouds
    cirrus_bit = 1 << 11   # cirrus
    mask = (qa.bitwiseAnd(cloud_bit).eq(0)
            .And(qa.bitwiseAnd(cirrus_bit).eq(0)))
    return (image.updateMask(mask).divide(10000)
            .copyProperties(image, ['system:time_start']))

def build_yearly_composite(year):
    """Cloud-free median Sentinel-2 composite for one year over the AOI."""
    start, end = f'{year}{SEASON_START}', f'{year}{SEASON_END}'
    collection = (ee.ImageCollection(S2_COLLECTION)
                  .filterBounds(aoi)
                  .filterDate(start, end)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',
                                       MAX_CLOUD_PERCENT))
                  .map(mask_s2_clouds))
    composite = collection.median().clip(aoi)
    return composite, collection

composite_before, coll_before = build_yearly_composite(YEAR_BEFORE)
composite_after,  coll_after  = build_yearly_composite(YEAR_AFTER)

# Sanity check: enough scenes in each year?
n_before = coll_before.size().getInfo()
n_after  = coll_after.size().getInfo()
print(f"Scenes used for {YEAR_BEFORE}: {n_before}")
print(f"Scenes used for {YEAR_AFTER}: {n_after}")
print("Composites built." if n_before and n_after
      else "WARNING: a year has no usable scenes - adjust config.")

Scenes used for 2019: 46
Scenes used for 2024: 27
Composites built.


## 4. NDVI and forest classification

NDVI = (NIR − Red) / (NIR + Red), using Sentinel-2 bands B8 and B4. Dense, healthy vegetation scores high. Forest is then every pixel above the NDVI threshold — one comparison, fully explainable.

In [5]:
def add_ndvi(composite):
    """Append an 'NDVI' band to a composite."""
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return composite.addBands(ndvi)

def classify_forest(composite, threshold):
    """Binary forest mask (1 = forest) from an NDVI threshold."""
    return composite.select('NDVI').gt(threshold).rename('forest')

composite_before = add_ndvi(composite_before)
composite_after  = add_ndvi(composite_after)

forest_before = classify_forest(composite_before, NDVI_FOREST_THRESHOLD)
forest_after  = classify_forest(composite_after,  NDVI_FOREST_THRESHOLD)

print(f"NDVI computed; forest classified at NDVI > {NDVI_FOREST_THRESHOLD}.")

NDVI computed; forest classified at NDVI > 0.6.


## 5. Visual validation

Before trusting any number, we check the classification against reality. Toggle **True colour** under the **Forest** layer: forest pixels should sit on visibly dense canopy and stop where the forest stops. This is how the 0.6 threshold was confirmed (non-forest zones inside the reserve are real settlement/cultivation, not missed canopy).

In [6]:
def build_validation_map():
    m = geemap.Map()
    m.centerObject(aoi, zoom=11)
    m.addLayer(composite_before, true_colour_vis, f'True colour {YEAR_BEFORE}')
    m.addLayer(composite_after,  true_colour_vis, f'True colour {YEAR_AFTER}')
    m.addLayer(composite_before.select('NDVI'), ndvi_vis, f'NDVI {YEAR_BEFORE}')
    m.addLayer(forest_before.selfMask(), {'palette': [FOREST_COLOR]},
               f'Forest {YEAR_BEFORE}')
    m.addLayer(forest_after.selfMask(), {'palette': [FOREST_COLOR]},
               f'Forest {YEAR_AFTER}')
    m.addLayer(ee.Image().paint(aoi, 0, 2), {'palette': 'black'}, 'AOI')
    m.add_layer_control()
    return m

build_validation_map()

Map(center=[-0.45042249774268284, 35.870543063024556], controls=(WidgetControl(options=['position', 'transpare…

## 6. Baseline normalization

The two composites can sit at slightly different NDVI baselines (different scene counts and dates), which would distort change detection. We measure that systematic offset over **stable forest** — land that didn't meaningfully change — and remove it from the later year.

For Eastern Mau, unchanged forest read ~0.047 NDVI higher in 2024; this corrects for that.

In [7]:
# Stable reference = forest in BOTH naive years (unchanged canopy).
stable_ref = forest_before.And(forest_after)

def median_ndvi_over(mask, composite, region, scale=10):
    """Median NDVI of a composite within a mask, over a region."""
    masked = composite.select('NDVI').updateMask(mask)
    stat = masked.reduceRegion(ee.Reducer.median(), region, scale,
                               maxPixels=1e13)
    return ee.Number(stat.get('NDVI'))

ref_2019 = median_ndvi_over(stable_ref, composite_before, aoi)
ref_2024 = median_ndvi_over(stable_ref, composite_after,  aoi)
ndvi_offset = ref_2024.subtract(ref_2019)

print(f"Stable-core median NDVI {YEAR_BEFORE}: {ref_2019.getInfo():.4f}")
print(f"Stable-core median NDVI {YEAR_AFTER}: {ref_2024.getInfo():.4f}")
print(f"Systematic offset ({YEAR_AFTER}-{YEAR_BEFORE}): {ndvi_offset.getInfo():+.4f}")

def apply_ndvi_offset(composite, offset):
    """Shift a composite's NDVI band down by `offset`."""
    corrected = composite.select('NDVI').subtract(offset).rename('NDVI')
    return composite.addBands(corrected, overwrite=True)

composite_after_corr = apply_ndvi_offset(composite_after, ndvi_offset)
print("2024 composite baseline-corrected.")

Stable-core median NDVI 2019: 0.7490
Stable-core median NDVI 2024: 0.7959
Systematic offset (2024-2019): +0.0469
2024 composite baseline-corrected.


## 7. Forest loss detection

We report **loss of established 2019 forest**, not gain. Loss is anchored to pixels we know were forest, so it is robust to baseline drift; gain is not reliably separable from inter-annual signal differences with this data, so we do not claim it.

Loss = was forest in 2019 **and** corrected NDVI dropped by at least the change threshold. Small patches (< 0.5 ha) are filtered as noise.

In [8]:
def remove_small_patches(binary_image, min_pixels):
    """Keep only connected groups of >= min_pixels; drop isolated noise."""
    patch_size = binary_image.selfMask().connectedPixelCount(
        maxSize=256, eightConnected=True)
    return binary_image.updateMask(patch_size.gte(min_pixels))

def build_loss_map(forest_baseline, comp_t1, comp_t2_corr, change_thresh):
    """Binary loss map within the established baseline forest."""
    delta = comp_t2_corr.select('NDVI').subtract(comp_t1.select('NDVI'))
    return forest_baseline.And(delta.lte(change_thresh * -1))

loss_raw   = build_loss_map(forest_before, composite_before,
                            composite_after_corr, NDVI_CHANGE_THRESHOLD)
loss_clean = remove_small_patches(loss_raw, MIN_PATCH_PIXELS)

# Final two-class map: 1 = forest retained, 2 = forest lost.
forest_status = (ee.Image(0)
                 .where(forest_before, 1)
                 .where(loss_clean, 2)
                 .rename('status').clip(aoi))
forest_status = forest_status.updateMask(forest_status.gt(0))
print("Loss map built and noise-filtered.")

Loss map built and noise-filtered.


## 8. Quantification

`pixelArea()` gives each pixel's true area; we sum it per class at native 10 m resolution and convert to hectares. The percentage is the decision-ready figure.

In [9]:
def area_by_class_ha(class_img, class_code, region, scale=10):
    """Area in hectares for one class code within a region (band-agnostic)."""
    class_mask = class_img.eq(class_code)
    area_image = (class_mask.multiply(ee.Image.pixelArea())
                  .rename('area'))
    stats = area_image.reduceRegion(ee.Reducer.sum(), region, scale,
                                    maxPixels=1e13)
    return ee.Number(stats.get('area')).divide(10000)

retained_ha = area_by_class_ha(forest_status, 1, aoi).getInfo()
lost_ha     = area_by_class_ha(forest_status, 2, aoi).getInfo()
baseline_ha = retained_ha + lost_ha
pct_lost    = (lost_ha / baseline_ha * 100) if baseline_ha else 0
annual_rate = pct_lost / (YEAR_AFTER - YEAR_BEFORE)

print("=" * 58)
print("  EASTERN MAU - FOREST LOSS FROM 2019 BASELINE")
print(f"  Period: {YEAR_BEFORE}-{YEAR_AFTER}  ({YEAR_AFTER-YEAR_BEFORE} years)")
print("=" * 58)
print(f"  Established forest, {YEAR_BEFORE} : {baseline_ha:>11,.1f} ha")
print(f"  Forest retained to {YEAR_AFTER}  : {retained_ha:>11,.1f} ha")
print(f"  Forest LOST              : {lost_ha:>11,.1f} ha  ({pct_lost:.2f}%)")
print(f"  Mean annual loss rate    : {annual_rate:>11,.2f} %/yr")
print("=" * 58)
print("  Note: gain/regrowth not reported - not reliably")
print("  separable from inter-annual composite differences.")

  EASTERN MAU - FOREST LOSS FROM 2019 BASELINE
  Period: 2019-2024  (5 years)
  Established forest, 2019 :    27,374.1 ha
  Forest retained to 2024  :    27,049.1 ha
  Forest LOST              :       325.0 ha  (1.19%)
  Mean annual loss rate    :        0.24 %/yr
  Note: gain/regrowth not reported - not reliably
  separable from inter-annual composite differences.


## 9. Final map and figure export

The loss map over a true-colour backdrop, plus a static PNG export for the README. Open the printed thumbnail URL and save it as `figures/change_map.png`, or screenshot the interactive map for a version with the basemap context.

In [10]:
status_vis = {'min': 1, 'max': 2, 'palette': [FOREST_COLOR, LOSS_COLOR]}

def build_final_map():
    m = geemap.Map()
    m.centerObject(aoi, zoom=11)
    m.addLayer(composite_after_corr, true_colour_vis, f'True colour {YEAR_AFTER}')
    m.addLayer(forest_status, status_vis, 'Forest status (retained / lost)')
    m.addLayer(ee.Image().paint(aoi, 0, 2), {'palette': 'black'}, 'AOI')
    legend = {'Forest retained': FOREST_COLOR.lstrip('#'),
              'Forest lost':     LOSS_COLOR.lstrip('#')}
    m.add_legend(title=f'Eastern Mau {YEAR_BEFORE}-{YEAR_AFTER}',
                 legend_dict=legend)
    m.add_layer_control()
    return m

# Static thumbnail for the README figure:
thumb_url = forest_status.getThumbURL({
    'min': 1, 'max': 2,
    'palette': [FOREST_COLOR.lstrip('#'), LOSS_COLOR.lstrip('#')],
    'region': aoi, 'dimensions': 1024})
print("Figure thumbnail (open in browser, save as figures/change_map.png):")
print(thumb_url)

build_final_map()

Figure thumbnail (open in browser, save as figures/change_map.png):
https://earthengine.googleapis.com/v1/projects/mau-sentinel/thumbnails/9827be190a7a6187c01b98d7094e7a53-891b4aaf5de066514136a73d81f878c3:getPixels


Map(center=[-0.45042249774268284, 35.870543063024556], controls=(WidgetControl(options=['position', 'transpare…